<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_11_Hybrid_Neyman_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 11 — A simulator-calibrated hybrid Neyman construction

Exercise 5 ended with pseudo-experiments from the learned hNDE model and the sampling distribution of the profile-likelihood-ratio statistic. We now amortize a Neyman construction over the signal strength $\mu$ and then calibrate it with a limited simulator budget.

The notebook deliberately separates four statistical objects:

1. the **frozen event-level hNDE likelihood**, which defines the ordering statistic $t_\mu$;
2. a conditional spline plus a first density-ratio correction, which learns the hNDE toy distribution $p_{\rm H}(t\mid\mu)$ from $500{,}000$ inexpensive hNDE pseudo-experiments;
3. a second density ratio learned from $100{,}000$ simulator pseudo-experiments, expressed in the hNDE probability-integral-transform (PIT) coordinate $u_0=F_{\rm H}(t\mid\mu)$;
4. a completely untouched $100{,}000$-toy simulator ensemble used only by an LF2I-style coverage auditor.

The second correction is still the simulator-to-hNDE density ratio. The PIT is an exact monotone change of coordinate that makes its reference distribution $\mathrm{Uniform}(0,1)$, eliminates the importance-weighted BCE used in the first version of this exercise, and leaves only a one-dimensional normalization integral even when the parameter vector is high-dimensional.

The expensive Exercise 5 event networks are frozen and loaded from their checkpoints. Toy ensembles are generated in resumable shards. The hNDE toys, statistic flow, and first ratio retain their original cache paths; the new PIT calibration has separate versioned paths, so a previous full Exercise 11 run can reuse its expensive first stage safely.


In [ ]:
## ==========================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ==========================================================================
import os, sys

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
    if REMAKE_EVENTS or not Path("dataframes/signal.parquet").exists():
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )

print("Working dir:", os.getcwd())


## Statistical construction at a glance

For a pseudo-dataset $\mathcal D$ and a tested value $\mu$, Exercise 5 uses

$$
t_\mu(\mathcal D)
=-2\log\frac{L_{\rm H}(\mu;\mathcal D)}
                 {L_{\rm H}(\widehat\mu;\mathcal D)}
\geq 0,
$$

where both the numerator and denominator use the same frozen hNDE likelihood. Simulator toys change the law that generates $\mathcal D$; they do **not** replace the fitted likelihood by the simulator truth.

First, a conditional spline $q_\phi$ and a matched classifier learn the hNDE toy density. In the transformed coordinate $y=\log(t+\epsilon)$,

$$
C_\phi(\mu)=\int_{\log\epsilon}^{\infty}q_\phi(v\mid\mu)\,dv,
\qquad
\widetilde q_\phi(y\mid\mu)
=\frac{q_\phi(y\mid\mu)\,\mathbb I[y\geq\log\epsilon]}
       {C_\phi(\mu)}
$$

is the physical, truncated flow reference. The first corrected density is

$$
p_{\rm H}(y\mid\mu)
=\frac{\widetilde q_\phi(y\mid\mu)r_1(y,\mu)}
       {\widetilde Z_1(\mu)},
\qquad
F_{\rm H}(t\mid\mu)
=\int_{\log\epsilon}^{\log(t+\epsilon)}p_{\rm H}(y\mid\mu)\,dy.
$$

A simulator toy is then mapped to

$$
u_0=F_{\rm H}(t_\mu\mid\mu)\in[0,1].
$$

If the hNDE toy law equaled the simulator law, $u_0\mid\mu$ would be uniform. We therefore train a second matched classifier for

$$
r_{\rm cal}(u,\mu)
=\frac{p_{\rm sim}^{U_0}(u\mid\mu)}{\mathrm{Uniform}(u)}
=p_{\rm sim}^{U_0}(u\mid\mu).
$$

Finite neural odds are normalized explicitly,

$$
G(u\mid\mu)
=\frac{\int_0^u r_{\rm cal}(v,\mu)\,dv}
       {\int_0^1 r_{\rm cal}(v,\mu)\,dv},
$$

giving the final calibrated conditional CDF and pivot

$$
F_{\rm cal}(t\mid\mu)
=G\!\left(F_{\rm H}(t\mid\mu)\mid\mu\right),
\qquad
U_{\rm cal}=F_{\rm cal}(T_\mu\mid\mu).
$$

In the oracle continuous limit, $U_{\rm cal}\mid\mu\sim\mathrm{Uniform}(0,1)$. Because large $t_\mu$ rejects the tested value, the 95% Neyman acceptance rule is $U_{\rm cal}\leq0.95$. Equivalently one may use $T_{\rm cal}=F^{-1}_{\chi^2_1}(U_{\rm cal})$ and compare it with $3.841$. The PIT convention is the upper-CDF convention appropriate to $t_\mu$; it reverses the lower-quantile sign convention used for the log-likelihood-ratio statistic in the [LF2I paper](https://arxiv.org/abs/2107.03920).


In [ ]:
import gc
import os
from pathlib import Path

# JAX fits the toy batches before PyTorch trains the large spline. Avoid
# reserving the whole Colab GPU so both frameworks can share it.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.stats import chi2

import torch

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_hnpe import (
    train_ratio_classifier,
    train_spline_flow,
)
from utils_neyman import (
    asimov_test_statistic,
    binned_coverage,
    build_compressed_q_model,
    conditional_cdf_values,
    conditional_density_grid,
    conditional_quantiles,
    conditional_ratio_grid,
    conditional_row_quantiles,
    conservative_empirical_quantile,
    coverage_auditor_probability,
    run_cached_toy_ensemble,
    sample_truncated_spline_flow,
    simulator_templates_from_exercise5,
    train_coverage_auditor,
    wilson_interval,
)
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    flow_sample_x,
    load_flow,
)
from utils_plotting import export_standalone_figure_script

FEATURES = list(FEATURES)
SEED = 11082026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Configuration and frozen Exercise 5 inputs

`FAST_MODE=True` is a structural test. The default construction uses 500,000 hNDE toys, 100,000 simulator-calibration toys, and a separate 100,000-toy audit. Checkpoints and toy shards persist in Drive.

Two cache namespaces are intentional:

- `BASE_RUN_TAG` points to the original hNDE toys, conditional statistic flow, and first ratio. Their statistical definition has not changed, so a completed first run can load them.
- `PIT_RUN_TAG` contains the pooled-template simulator toys, PIT-ratio ensemble, numerical calibration map, and auditor. These objects must not load the old raw-$t$, importance-weighted second correction.

The statistic flow uses the Exercise 5 rational-quadratic-spline settings: ten spline transforms, 16 bins, tail bound 5, four residual blocks, and width 1024. For a scalar target, `utils_hnpe` uses a one-dimensional autoregressive spline; a coupling layer would otherwise have no second coordinate to transform.


In [ ]:
BASE_PATH = Path("dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
HYBRID_DENSITY_DIR = Path("saved_densities_hybrid")
EXERCISE5_RATIO_NORMALIZATION_PATH = (
    HYBRID_DENSITY_DIR / "ratio_normalization.npz"
)
if EXERCISE5_RATIO_NORMALIZATION_PATH.exists():
    with np.load(EXERCISE5_RATIO_NORMALIZATION_PATH) as saved_norm:
        EXERCISE5_RATIO_NORMALIZATION = {
            "signal": float(saved_norm["signal"]),
            "background": float(saved_norm["background"]),
        }
    EXERCISE5_NORMALIZATION_SOURCE = str(
        EXERCISE5_RATIO_NORMALIZATION_PATH
    )
else:
    # Legacy Exercise 5 saved normalized ratio arrays but not these two
    # finite-reference means. These values are transcribed from the
    # committed full-run output (printed to six decimal places).
    EXERCISE5_RATIO_NORMALIZATION = {
        "signal": 0.999149,
        "background": 0.999669,
    }
    EXERCISE5_NORMALIZATION_SOURCE = "committed Exercise 5 output"

FAST_MODE = False
LOAD_IF_AVAILABLE = True
BASE_RUN_TAG = "fast" if FAST_MODE else "full_v1"
PIT_RUN_TAG = "fast_pit_v1" if FAST_MODE else "full_pit_v1"
MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / BASE_RUN_TAG
PIT_MODEL_DIR = Path("models_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PIT_CACHE_DIR = Path("saved_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
PLOT_DIR = Path("plots_exercise11_hybrid_neyman_v1") / PIT_RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise11_figures_scripts")
RATIO1_MODEL_DIR = MODEL_DIR / "hnde_residual_ensemble4"
CALIBRATION_RATIO_MODEL_DIR = PIT_MODEL_DIR / "pit_calibration_ensemble4"
for directory in [
    MODEL_DIR, CACHE_DIR, PIT_MODEL_DIR, PIT_CACHE_DIR,
    PLOT_DIR, FIGURE_SCRIPT_DIR,
    RATIO1_MODEL_DIR, CALIBRATION_RATIO_MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12

MU_RANGE = (0.0, 3.0)
TOY_Q_BINS = 512
TOY_BATCH_SIZE = 2_000
TOY_NEWTON_STEPS = 16
TOY_MU_MAX = 12.0
TOY_FIT_FINGERPRINT = "jax_newton16_bisection64_mumax12_v1"
T_OFFSET = 1.0e-6
LOG_RATIO_CLIP = 15.0
PIT_EPS = 1.0e-6
QUANTILE_LEVELS = np.asarray([0.50, 0.68, 0.90, 0.95, 0.99])
ANCHOR_MUS = np.asarray([0.0, 3.0])

if FAST_MODE:
    N_REFERENCE_EVENTS = 250_000
    N_HNDE_TOYS = 50_000
    N_FLOW_TOYS = 40_000
    N_RATIO1_TOYS = 10_000
    N_SIMULATOR_CALIBRATION_TOYS = 20_000
    N_SIMULATOR_AUDIT_TOYS = 20_000
    N_ANCHOR_TOYS = 5_000
    N_AUDIT_ANCHOR_TOYS = 5_000
    FLOW_EPOCHS = 8
    RATIO_EPOCHS = 12
    QUADRATURE_MU_POINTS = 101
    QUADRATURE_Y_POINTS = 1_025
    PIT_QUADRATURE_POINTS = 1_025
else:
    N_REFERENCE_EVENTS = 5_000_000
    N_HNDE_TOYS = 500_000
    N_FLOW_TOYS = 400_000
    N_RATIO1_TOYS = 100_000
    N_SIMULATOR_CALIBRATION_TOYS = 100_000
    N_SIMULATOR_AUDIT_TOYS = 100_000
    N_ANCHOR_TOYS = 25_000
    N_AUDIT_ANCHOR_TOYS = 25_000
    FLOW_EPOCHS = 70
    RATIO_EPOCHS = 50
    QUADRATURE_MU_POINTS = 301
    QUADRATURE_Y_POINTS = 4_097
    PIT_QUADRATURE_POINTS = 4_097

if N_FLOW_TOYS + N_RATIO1_TOYS != N_HNDE_TOYS:
    raise ValueError("The disjoint hNDE flow/ratio splits must exhaust the toys.")

STATISTIC_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
STATISTIC_FLOW_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 5,
    "gradient_clip": 5.0,
}
CORRECTION_MODEL_CONFIG = {
    "hidden_features": 1024,
    "hidden_layers": 4,
    "dropout_probability": 0.0,
}
CORRECTION_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": RATIO_EPOCHS,
    "learning_rate": 1.0e-3,
    "lr_scheduler": "step",
    "lr_scheduler_factor": 0.01,
    "lr_scheduler_patience": 10,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}
AUDITOR_MODEL_CONFIG = {"hidden_features": 64, "hidden_layers": 3}
AUDITOR_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": 200 if not FAST_MODE else 40,
    "learning_rate": 1.0e-3,
    "weight_decay": 1.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 5,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.20,
    "patience": 20,
    "gradient_clip": 5.0,
}


def export_exercise11_figure(fig, script_name):
    fig.savefig(PLOT_DIR / f"{script_name}.png", dpi=160)
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
    HYBRID_DENSITY_DIR / "weights_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend([
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        ])
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Exercise 11 loads the frozen Exercise 5 model and held-out arrays. "
        "Run Exercise 5 through 'Reconstruct and validate the hybrid densities' "
        "first. Missing:\n" + "\n".join(f"  - {path}" for path in missing_paths)
    )

print(f"Base/PIT run tags: {BASE_RUN_TAG} / {PIT_RUN_TAG}")
print(f"hNDE toys: {N_HNDE_TOYS:,}")
print(f"simulator calibration/audit toys: "
      f"{N_SIMULATOR_CALIBRATION_TOYS:,}/{N_SIMULATOR_AUDIT_TOYS:,}")
print(f"Standalone figure scripts: {FIGURE_SCRIPT_DIR}/")


## Load the frozen hNDE event model

The PRESEL classifier, post-selection yields, reference flow, and the two four-member density-ratio ensembles are the same objects used by Exercise 5. No event-level model is retrained here.


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_CANDIDATES = [
    CACHE_DIR / "exercise5_preselection_state.npz",
    Path("saved_asimov_nis_influence_v2/exercise5_preselection_state.npz"),
    Path("saved_exercise7_misspecification/exercise5_preselection_state.npz"),
]
existing_state = next(
    (path for path in PRESEL_STATE_CANDIDATES if path.exists()), None
)
if existing_state is not None:
    state = np.load(existing_state)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded PRESEL state from {existing_state}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms, statistics = {}, {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )
    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"], histograms["background"], edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_CANDIDATES[0],
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )

reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)
ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")


## Reconstruct and cache the compressed Exercise 5 likelihood

For every event, the fitted likelihood depends only on

$$
q(x)=\frac{\lambda_S r_S(x)}{\lambda_B r_B(x)}.
$$

We draw the same five-million-event reference sample as Exercise 5, normalize both process ratios on it, and compress $\log q$ into 512 bins. The compressed signal/background probabilities generate hNDE toys; their ratio defines the **frozen likelihood** used to fit both hNDE and simulator toys.


In [ ]:
def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(
            values[start : start + int(batch_size)], columns=FEATURES
        )
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch, scaler=pack["scaler"], model=pack["model"]
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"Non-finite {sample_name} ratio.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


COMPRESSED_MODEL_PATH = CACHE_DIR / "compressed_exercise5_model.npz"
compression_cache_metadata = {
    "cache_version": 1,
    "n_reference_events": N_REFERENCE_EVENTS,
    "toy_q_bins": TOY_Q_BINS,
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
    "presel_ratio_cut": PRESEL_RATIO_CUT,
}
if COMPRESSED_MODEL_PATH.exists():
    saved = np.load(COMPRESSED_MODEL_PATH)
    missing_metadata = set(compression_cache_metadata) - set(saved.files)
    mismatched_metadata = [
        name for name, expected in compression_cache_metadata.items()
        if name in saved.files
        and not np.isclose(float(saved[name]), float(expected), rtol=1e-12)
    ]
    if missing_metadata or mismatched_metadata:
        raise RuntimeError(
            "The compressed-model cache has stale provenance. Bump "
            "BASE_RUN_TAG (recommended) or remove only that versioned cache. "
            f"Missing={sorted(missing_metadata)}, "
            f"mismatched={mismatched_metadata}."
        )
    COMPRESSED_Q = saved["q"]
    HNDE_SIGNAL_PROBABILITY = saved["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = saved["background_probability"]
    LOG_Q_EDGES = saved["log_q_edges"]
    RATIO_NORMALIZATION = {
        "signal": float(saved["signal_normalization"]),
        "background": float(saved["background_normalization"]),
    }
    compression_truth = saved["validation_truth"]
    compression_test = saved["validation_test"]
    compression_unbinned = saved["validation_unbinned"]
    compression_binned = saved["validation_binned"]
    saved.close()
    print(f"Loaded compressed hNDE model from {COMPRESSED_MODEL_PATH}")
else:
    torch.manual_seed(SEED + 100)
    reference_values, reference_acceptance = sample_preselected_flow(
        reference_flow, N_REFERENCE_EVENTS, REFERENCE_SAMPLING_BATCH_SIZE
    )
    raw_signal = evaluate_ratio("signal", reference_values)
    raw_background = evaluate_ratio("background", reference_values)
    RATIO_NORMALIZATION = {
        "signal": float(raw_signal.mean()),
        "background": float(raw_background.mean()),
    }
    ratio_signal = raw_signal / RATIO_NORMALIZATION["signal"]
    ratio_background = raw_background / RATIO_NORMALIZATION["background"]
    weight_signal = ratio_signal / N_REFERENCE_EVENTS
    weight_background = ratio_background / N_REFERENCE_EVENTS
    event_q = (
        LAM_SIG / LAM_BKG * ratio_signal / ratio_background
    )
    compressed = build_compressed_q_model(
        event_q,
        weight_signal,
        weight_background,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        n_bins=TOY_Q_BINS,
    )
    COMPRESSED_Q = compressed["q"]
    HNDE_SIGNAL_PROBABILITY = compressed["signal_probability"]
    HNDE_BACKGROUND_PROBABILITY = compressed["background_probability"]
    LOG_Q_EDGES = compressed["log_q_edges"]

    compression_rows = []
    for truth_mu in [0.0, 0.25, 1.0, 2.0, 3.0]:
        unbinned_expected = (
            truth_mu * LAM_SIG * weight_signal
            + LAM_BKG * weight_background
        )
        binned_expected = (
            truth_mu * LAM_SIG * HNDE_SIGNAL_PROBABILITY
            + LAM_BKG * HNDE_BACKGROUND_PROBABILITY
        )
        for test_mu in [0.0, 0.5, 1.0, 2.0, 3.0]:
            if test_mu == truth_mu:
                continue
            compression_rows.append((
                truth_mu,
                test_mu,
                asimov_test_statistic(
                    test_mu, truth_mu, event_q, unbinned_expected,
                    lam_signal=LAM_SIG,
                ),
                asimov_test_statistic(
                    test_mu, truth_mu, COMPRESSED_Q, binned_expected,
                    lam_signal=LAM_SIG,
                ),
            ))
    compression_rows = np.asarray(compression_rows, dtype=np.float64)
    compression_truth = compression_rows[:, 0]
    compression_test = compression_rows[:, 1]
    compression_unbinned = compression_rows[:, 2]
    compression_binned = compression_rows[:, 3]
    np.savez_compressed(
        COMPRESSED_MODEL_PATH,
        q=COMPRESSED_Q,
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        log_q_edges=LOG_Q_EDGES,
        signal_normalization=RATIO_NORMALIZATION["signal"],
        background_normalization=RATIO_NORMALIZATION["background"],
        validation_truth=compression_truth,
        validation_test=compression_test,
        validation_unbinned=compression_unbinned,
        validation_binned=compression_binned,
        **compression_cache_metadata,
    )
    print(f"Saved compressed hNDE model to {COMPRESSED_MODEL_PATH}")
    print(f"Reference PRESEL acceptance: {reference_acceptance:.3%}")
    del reference_values, raw_signal, raw_background
    del ratio_signal, ratio_background, weight_signal, weight_background, event_q
    gc.collect()

print("Ratio normalizations:", RATIO_NORMALIZATION)
print("Compressed q quantiles:", np.quantile(COMPRESSED_Q, [0, .01, .5, .99, 1]))

# The frozen event networks are no longer needed after compression. Free
# their Torch/ONNX GPU allocations before the large statistic flow trains.
reference_flow = None
ratio_models = None
PRESEL_model = None
PRESEL_scaler = None
model_proto = None
scaler = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released frozen Exercise 5 event networks.")


### Validate the compression over the full design interval

Exercise 5 checked one Asimov displacement. Here the unbinned and compressed expected statistics are compared for several generating and tested values across $[0,3]$. This validates the numerical acceleration before it is used half a million times.


In [ ]:
compression_validation = pd.DataFrame({
    "mu_true": compression_truth,
    "mu_test": compression_test,
    "t_unbinned": compression_unbinned,
    "t_compressed": compression_binned,
})
compression_validation["absolute_difference"] = (
    compression_validation["t_compressed"]
    - compression_validation["t_unbinned"]
)
compression_validation["relative_difference"] = np.divide(
    compression_validation["absolute_difference"],
    compression_validation["t_unbinned"],
    out=np.zeros(len(compression_validation)),
    where=compression_validation["t_unbinned"] > 1.0e-8,
)
display(compression_validation.style.format(precision=6).hide(axis="index"))
max_relative = float(compression_validation["relative_difference"].abs().max())
max_absolute = float(compression_validation["absolute_difference"].abs().max())
print(f"Maximum relative/absolute change: {max_relative:.3%} / {max_absolute:.4g}")
if max_relative > 5.0e-3 and max_absolute > 2.0e-2:
    raise RuntimeError(
        "The 512-bin compression is not accurate to 0.5% across the scan. "
        "Increase TOY_Q_BINS."
    )


## Vectorized pseudo-experiment fits

For a binned pseudo-experiment with counts $n_j$, the constrained MLE solves

$$
0=\lambda_S-\sum_j n_j\frac{q_j}{1+\widehat\mu q_j},
\qquad \widehat\mu\geq0.
$$

JAX performs 16 bounded Newton steps for an entire toy batch. The statistic is then

$$
t_\mu=2\left[(\mu-\widehat\mu)\lambda_S
-\sum_j n_j\left\{\log(1+\mu q_j)-\log(1+\widehat\mu q_j)\right\}\right].
$$

Only one combined Poisson draw is needed per bin, with mean $\mu\lambda_S P_{S,j}+\lambda_B P_{B,j}$. The $2000\times512$ count matrix is discarded after each fit.


In [ ]:
COMPRESSED_Q_JAX = jnp.asarray(COMPRESSED_Q)


@jax.jit
def fit_compressed_toy_batch(counts, test_mu):
    counts = jnp.asarray(counts, dtype=jnp.float64)
    test_mu = jnp.asarray(test_mu, dtype=jnp.float64).reshape(-1)
    q_values = COMPRESSED_Q_JAX
    initial_mu = jnp.clip(
        (jnp.sum(counts, axis=1) - LAM_BKG) / LAM_SIG,
        0.0,
        TOY_MU_MAX,
    )
    score_at_zero = LAM_SIG - jnp.sum(counts * q_values, axis=1)

    def newton_step(_, mu):
        response = q_values / (1.0 + mu[:, None] * q_values)
        score = LAM_SIG - jnp.sum(counts * response, axis=1)
        information = jnp.sum(counts * response**2, axis=1)
        step = jnp.clip(
            score / jnp.maximum(information, 1.0e-12), -2.0, 2.0
        )
        return jnp.clip(mu - step, 0.0, TOY_MU_MAX)

    mu_hat = jax.lax.fori_loop(
        0, TOY_NEWTON_STEPS, newton_step, initial_mu
    )
    mu_hat = jnp.where(score_at_zero >= 0.0, 0.0, mu_hat)
    statistic = 2.0 * (
        (test_mu - mu_hat) * LAM_SIG
        - jnp.sum(
            counts * (
                jnp.log1p(test_mu[:, None] * q_values)
                - jnp.log1p(mu_hat[:, None] * q_values)
            ),
            axis=1,
        )
    )
    fitted_response = q_values / (1.0 + mu_hat[:, None] * q_values)
    fitted_score = LAM_SIG - jnp.sum(
        counts * fitted_response, axis=1
    )
    fitted_score = jnp.where(mu_hat == 0.0, 0.0, fitted_score)
    return mu_hat, jnp.maximum(statistic, 0.0), fitted_score


def fit_toy_batch_numpy(counts, test_mu):
    counts = np.asarray(counts)
    test_mu = np.asarray(test_mu, dtype=np.float64).reshape(-1)
    result = fit_compressed_toy_batch(counts, test_mu)
    mu_hat, t_mu, fitted_score = [
        np.asarray(values, dtype=np.float64) for values in result
    ]
    failed = (
        (mu_hat > 1.0e-10)
        & (mu_hat < TOY_MU_MAX - 1.0e-10)
        & (np.abs(fitted_score) > 1.0e-6)
    )
    if np.any(failed):
        # The score is monotone increasing in mu. A vectorized bisection
        # fallback makes rare Newton failures harmless without returning
        # to one-Minuit-fit-per-toy execution.
        failed_counts = counts[failed].astype(np.float64)
        lower = np.zeros(np.sum(failed), dtype=np.float64)
        upper = np.full(np.sum(failed), TOY_MU_MAX, dtype=np.float64)
        for _ in range(64):
            middle = 0.5 * (lower + upper)
            response = COMPRESSED_Q / (
                1.0 + middle[:, None] * COMPRESSED_Q
            )
            middle_score = LAM_SIG - np.sum(
                failed_counts * response, axis=1
            )
            move_lower = middle_score < 0.0
            lower = np.where(move_lower, middle, lower)
            upper = np.where(move_lower, upper, middle)
        repaired_mu = 0.5 * (lower + upper)
        mu_hat[failed] = repaired_mu
        failed_test_mu = test_mu[failed]
        repaired_t = 2.0 * (
            (failed_test_mu - repaired_mu) * LAM_SIG
            - np.sum(
                failed_counts
                * (
                    np.log1p(failed_test_mu[:, None] * COMPRESSED_Q)
                    - np.log1p(repaired_mu[:, None] * COMPRESSED_Q)
                ),
                axis=1,
            )
        )
        t_mu[failed] = np.maximum(repaired_t, 0.0)
        repaired_response = COMPRESSED_Q / (
            1.0 + repaired_mu[:, None] * COMPRESSED_Q
        )
        fitted_score[failed] = LAM_SIG - np.sum(
            failed_counts * repaired_response, axis=1
        )
    if np.any(mu_hat >= TOY_MU_MAX - 1.0e-10):
        raise RuntimeError(
            "A toy MLE reached TOY_MU_MAX; increase the fit bound."
        )
    return mu_hat, t_mu, fitted_score


# Compile once and check that the fitted score is small away from the boundary.
_rng = np.random.default_rng(SEED + 200)
_mu = _rng.uniform(*MU_RANGE, size=8)
_mean = (
    _mu[:, None] * LAM_SIG * HNDE_SIGNAL_PROBABILITY[None, :]
    + LAM_BKG * HNDE_BACKGROUND_PROBABILITY[None, :]
)
_counts = _rng.poisson(_mean)
_muhat, _tmu, _score = fit_toy_batch_numpy(_counts, _mu)
print("Toy-kernel smoke test:")
print("  mu_true:", np.round(_mu, 3))
print("  mu_hat: ", np.round(_muhat, 3))
print("  t_mu:  ", np.round(_tmu, 3))
print(f"  max interior score residual: {np.max(np.abs(_score)):.3e}")
del _rng, _mu, _mean, _counts, _muhat, _tmu, _score


## 1. Generate 500,000 hNDE toys over $\mu\sim U(0,3)$

The ensemble is divided *before training*: 400,000 rows train/validate the conditional flow, and the remaining 100,000 rows train the first matched residual. This avoids asking a classifier to correct a flow on the same pseudo-experiments that fitted the flow.


In [ ]:
hnde_toys = run_cached_toy_ensemble(
    cache_dir=CACHE_DIR / f"hnde_uniform_{N_HNDE_TOYS}",
    n_toys=N_HNDE_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 300,
    mu_range=MU_RANGE,
    signal_probability=HNDE_SIGNAL_PROBABILITY,
    background_probability=HNDE_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
print(pd.DataFrame({
    "mu": hnde_toys["mu"],
    "mu_hat": hnde_toys["mu_hat"],
    "t_mu": hnde_toys["t_mu"],
    "n_events": hnde_toys["n_events"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())
print(
    "Maximum fitted-score residual:",
    f"{np.max(np.abs(hnde_toys['fitted_score'])):.3e}",
)

split_rng = np.random.default_rng(SEED + 301)
toy_order = split_rng.permutation(N_HNDE_TOYS)
flow_indices = toy_order[:N_FLOW_TOYS]
ratio1_indices = toy_order[N_FLOW_TOYS:]
flow_mu = hnde_toys["mu"][flow_indices].astype(np.float32)
flow_y = np.log(
    hnde_toys["t_mu"][flow_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)
ratio1_mu = hnde_toys["mu"][ratio1_indices].astype(np.float32)
ratio1_y_positive = np.log(
    hnde_toys["t_mu"][ratio1_indices].astype(np.float64) + T_OFFSET
).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].hexbin(
    hnde_toys["mu"], hnde_toys["mu_hat"],
    gridsize=80, bins="log", mincnt=1, cmap="viridis",
)
axes[0].plot(MU_RANGE, MU_RANGE, "w--", lw=1.4)
axes[0].set(xlabel=r"$\mu_{\rm true}$", ylabel=r"$\widehat\mu$",
            title="Amortized hNDE pseudo-experiments")
for low, high, color in [(0.0,.25,"C0"),(.75,1.0,"C1"),(1.75,2.0,"C2"),(2.75,3.0,"C3")]:
    mask = (hnde_toys["mu"] >= low) & (hnde_toys["mu"] < high)
    values = hnde_toys["t_mu"][mask]
    edges = np.linspace(0, np.quantile(values, .995), 60)
    axes[1].hist(values, bins=edges, density=True, histtype="step", lw=1.8,
                 color=color, label=rf"$\mu\in[{low:g},{high:g})$")
x = np.linspace(0.001, axes[1].get_xlim()[1], 400)
axes[1].plot(x, chi2.pdf(x, df=1), "k--", lw=1.3, label=r"$\chi^2_1$")
axes[1].set(xlabel=r"$t_{\mu_{\rm true}}$", ylabel="Density",
            title="The statistic is not assumed pivotal")
axes[1].legend(fontsize=8)
fig.tight_layout()
export_exercise11_figure(fig, "hnde_amortized_toys")
plt.show()


## 2. Train the conditional quadratic-spline reference

The monotone transformation

$$
y=\log(t_\mu+\epsilon),\qquad \epsilon=10^{-6},
$$

makes the positive, long-tailed statistic easier to model on the real line. All density ratios are trained in $(\mu,y)$ coordinates. The transformation is one-to-one for $t\geq0$, so its Jacobian cancels in the ratios and its conditional quantiles transform back exactly.


In [ ]:
statistic_flow = train_spline_flow(
    flow_y[:, None],
    context=flow_mu[:, None],
    checkpoint=MODEL_DIR / "q_phi_y_given_mu.pt",
    model_config=STATISTIC_FLOW_MODEL_CONFIG,
    training_config=STATISTIC_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 400,
    load_if_available=LOAD_IF_AVAILABLE,
)
flow_config_mismatch = {
    name: (statistic_flow["config"].get(name), expected)
    for name, expected in STATISTIC_FLOW_MODEL_CONFIG.items()
    if statistic_flow["config"].get(name) != expected
}
if flow_config_mismatch:
    raise RuntimeError(
        "The loaded statistic-flow checkpoint has stale architecture: "
        f"{flow_config_mismatch}. Bump BASE_RUN_TAG."
    )
print("Conditional statistic flow:", statistic_flow["checkpoint"])


## 3. First matched correction: flow $\rightarrow$ hNDE toys

For every held-out hNDE pair $(\mu_i,y_i^+)$, draw $y_i^-\sim\widetilde q_\phi(y\mid\mu_i)$ from the physical, truncated flow. The two classifier rows share exactly the same $\mu_i$ and remain in the same train/validation group. The arithmetic mean of four classifier odds estimates $r_1=p_{\rm H}/\widetilde q_\phi$.

The flow has real support while physical $y$ obeys $y\geq\log\epsilon$. Reference draws below that boundary are rejection-sampled. Their fraction is printed as a direct support diagnostic.


In [ ]:
Y_MIN = float(np.log(T_OFFSET))
ratio1_y_negative, ratio1_rejection = sample_truncated_spline_flow(
    statistic_flow,
    ratio1_mu[:, None],
    lower_bound=Y_MIN,
    seed=SEED + 500,
)
ratio1_positive = np.column_stack([ratio1_mu, ratio1_y_positive])
ratio1_negative = np.column_stack([ratio1_mu, ratio1_y_negative])
paired_ids_1 = np.arange(N_RATIO1_TOYS, dtype=np.int64)
ratio1_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(f"Training hNDE residual member {member + 1}/{RATIO_ENSEMBLE_SIZE}")
    print("=" * 76)
    ratio1_ensemble.append(
        train_ratio_classifier(
            ratio1_positive,
            ratio1_negative,
            checkpoint=RATIO1_MODEL_DIR / f"r1_member{member}.pt",
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 510 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_ids_1,
        )
    )
print(f"Unphysical flow-reference rejection fraction: {ratio1_rejection:.4%}")
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in ratio1_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(ratio1_ensemble):
    history = pack.get("history", {})
    ax.plot(history.get("validation", []), label=f"member {member}")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="First conditional-ratio ensemble")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "first_ratio_training")
plt.show()


## 4. Normalize the first hybrid density and compute $F_{\rm H}$

A finite classifier does not guarantee $\int q_\phi r_1\,dy=1$ at every $\mu$. We therefore form the normalized hNDE-level hybrid CDF

$$
F_{\rm H}(y\mid\mu)=
\frac{\int_{\log\epsilon}^{y}q_\phi(v\mid\mu)r_1(v,\mu)\,dv}
     {\int_{\log\epsilon}^{\infty}q_\phi(v\mid\mu)r_1(v,\mu)\,dv}
$$

by deterministic cumulative trapezoidal quadrature. This is the numerical primitive of the corrected density; after multiplying a flow by a neural ratio, the flow's analytic CDF is no longer the desired CDF.

The raw flow has real support, while physical $y$ obeys $y\geq\log\epsilon$. Its physical mass $C_\phi(\mu)$ is integrated separately so that the rejection-sampling diagnostic and the ratio normalization refer to the same base measure. The upper $y$ boundary is chosen from the hNDE toys and expanded adaptively until both the raw flow and the ratio-corrected hybrid have a small endpoint density and little probability in the final one-unit buffer. The raw-flow check is needed because rejection sampling draws from that law before $r_1$ is applied. The buffer-mass check is also important: a density can look small at one endpoint while a non-negligible integrated tail remains. We abort if the support does not become quiet, then repeat the CDF inversion on the nested half-resolution grid and require the learned quantiles to be stable.

After simulator toys are generated we impose the stricter support check that **no** calibration statistic lies beyond this frozen grid. A value outside the grid would be mapped artificially to $u_0=0$ or $1$, creating a fake atom that an ordinary density ratio cannot represent.


In [ ]:
MU_DENSITY_GRID = np.linspace(*MU_RANGE, QUADRATURE_MU_POINTS)
Y_MAX = float(max(
    5.0,
    np.quantile(np.concatenate([flow_y, ratio1_y_positive]), .99999) + 1.5,
))
Y_SUPPORT_BUFFER = 1.0
Y_SUPPORT_STEP = 1.0
Y_SUPPORT_MAX_EXPANSIONS = 3
Y_EDGE_RATIO_TARGET = 1.0e-5
Y_UPPER_BUFFER_MASS_TARGET = 1.0e-4

for support_attempt in range(Y_SUPPORT_MAX_EXPANSIONS + 1):
    Y_GRID = np.linspace(Y_MIN, Y_MAX, QUADRATURE_Y_POINTS)
    flow_physical_grid = conditional_density_grid(
        statistic_flow,
        [],
        MU_DENSITY_GRID,
        Y_GRID,
    )
    hybrid1_grid = conditional_density_grid(
        statistic_flow,
        [ratio1_ensemble],
        MU_DENSITY_GRID,
        Y_GRID,
        max_abs_log_ratio=LOG_RATIO_CLIP,
    )
    hybrid1_edge_ratio = float(np.max(
        hybrid1_grid["density"][:, -1]
        / np.max(hybrid1_grid["density"], axis=1)
    ))
    flow_edge_ratio = float(np.max(
        flow_physical_grid["density"][:, -1]
        / np.max(flow_physical_grid["density"], axis=1)
    ))
    upper_buffer_index = int(np.searchsorted(
        Y_GRID, Y_MAX - Y_SUPPORT_BUFFER
    ))
    hybrid1_upper_buffer_mass = float(np.max(
        1.0 - hybrid1_grid["cdf"][:, upper_buffer_index]
    ))
    flow_upper_buffer_mass = float(np.max(
        1.0 - flow_physical_grid["cdf"][:, upper_buffer_index]
    ))
    print(
        f"Y support attempt {support_attempt + 1}: Y_MAX={Y_MAX:.3f}, "
        f"hybrid edge/buffer={hybrid1_edge_ratio:.3e}/"
        f"{hybrid1_upper_buffer_mass:.3e}, raw-flow edge/buffer="
        f"{flow_edge_ratio:.3e}/{flow_upper_buffer_mass:.3e}"
    )
    support_is_quiet = (
        hybrid1_edge_ratio <= Y_EDGE_RATIO_TARGET
        and hybrid1_upper_buffer_mass <= Y_UPPER_BUFFER_MASS_TARGET
        and flow_edge_ratio <= Y_EDGE_RATIO_TARGET
        and flow_upper_buffer_mass <= Y_UPPER_BUFFER_MASS_TARGET
    )
    if support_is_quiet:
        break
    if support_attempt == Y_SUPPORT_MAX_EXPANSIONS:
        raise RuntimeError(
            "Y support did not converge. Increase "
            "Y_SUPPORT_MAX_EXPANSIONS or the initial Y_MAX, and inspect "
            "the raw-flow and r1 tails before continuing."
        )
    Y_MAX += Y_SUPPORT_STEP

hybrid1_quantile_y = conditional_quantiles(
    hybrid1_grid["cdf"], Y_GRID, QUANTILE_LEVELS
)
hybrid1_quantile_t = np.maximum(
    np.exp(hybrid1_quantile_y) - T_OFFSET, 0.0
)
hybrid1_log_normalization_truncated = (
    hybrid1_grid["log_normalization"]
    - flow_physical_grid["log_normalization"]
)

flow_physical_mass = np.exp(flow_physical_grid["log_normalization"])
if np.any((flow_physical_mass <= 0.0) | (flow_physical_mass > 1.02)):
    raise RuntimeError("Invalid numerical physical-support mass.")
if np.max(flow_physical_mass) > 1.001:
    print(
        "WARNING: flow quadrature exceeds unit mass by more than 0.1%; "
        "the half-resolution stability check below is especially important."
    )
ratio1_acceptance = np.interp(
    ratio1_mu,
    MU_DENSITY_GRID,
    np.clip(flow_physical_mass, 1.0e-12, 1.0),
)
quadrature_rejection = 1.0 - (
    len(ratio1_acceptance) / np.sum(1.0 / ratio1_acceptance)
)
print(
    "Physical flow mass C_phi(mu) range:",
    flow_physical_mass.min(), flow_physical_mass.max(),
)
print(
    "Flow rejection: observed / context-matched quadrature =",
    f"{ratio1_rejection:.4%} / {quadrature_rejection:.4%}",
)
if abs(ratio1_rejection - quadrature_rejection) > 5.0e-3:
    raise RuntimeError(
        "Rejection sampling and the numerical physical-support mass "
        "disagree; enlarge or refine Y_GRID."
    )
print(
    "log Z1_tilde(mu) quantiles:",
    np.quantile(
        hybrid1_log_normalization_truncated, [0, .01, .5, .99, 1]
    ),
)
print(
    "r1 quadrature logit range / clipped fraction:",
    hybrid1_grid["ratio_log_range"][0],
    f"{hybrid1_grid['ratio_clip_fraction'][0]:.4%}",
)
coarse_y_grid = Y_GRID[::2]
coarse_hybrid1_density = hybrid1_grid["density"][:, ::2]
coarse_hybrid1_normalization = trapezoid(
    coarse_hybrid1_density, x=coarse_y_grid, axis=1
)
coarse_hybrid1_cdf = cumulative_trapezoid(
    coarse_hybrid1_density, x=coarse_y_grid, axis=1, initial=0.0
) / coarse_hybrid1_normalization[:, None]
coarse_hybrid1_quantile_y = conditional_quantiles(
    coarse_hybrid1_cdf, coarse_y_grid, QUANTILE_LEVELS
)
y_resolution_shift = float(np.max(np.abs(
    coarse_hybrid1_quantile_y - hybrid1_quantile_y
)))
y_resolution_tolerance = max(0.02, 4.0 * np.max(np.diff(Y_GRID)))
print(
    "Full/half y-grid maximum quantile shift / tolerance:",
    y_resolution_shift, y_resolution_tolerance,
)
if y_resolution_shift > y_resolution_tolerance:
    raise RuntimeError(
        "The hNDE conditional quantiles are not stable when the y-grid "
        "resolution is halved. Increase QUADRATURE_Y_POINTS."
    )

for name, cdf in [("F_H", hybrid1_grid["cdf"])]:
    if (
        np.max(np.abs(cdf[:, 0])) > 1.0e-10
        or np.max(np.abs(cdf[:, -1] - 1.0)) > 1.0e-10
        or np.min(np.diff(cdf, axis=1)) < -1.0e-10
    ):
        raise RuntimeError(f"{name} is not a valid conditional CDF.")


## 5. Define one fixed simulator law for calibration and audit

Exercise 5 saved event ratios on final held-out simulator reservoirs—background first, then signal. These events were not used to train PRESEL, the reference flow, or either event-level ratio. We histogram **all** held-out events of each process into the frozen $\log q$ bins and treat the resulting pooled empirical probabilities as one fixed simulator law.

Calibration and audit pseudo-experiments use different random seeds but draw from this same pooled law. Sharing the fixed template is not leakage: the random toy datasets remain independent, while the template defines what “simulator truth” means in this numerical experiment. Splitting the finite event reservoir into two half-templates, as the first notebook version did, creates two detectably different data-generating laws. A cutoff calibrated under one law cannot be expected to cover under the other.

We still build the half-template probabilities and print their $L_1$ differences. They are now a **diagnostic of finite-simulator template uncertainty**, not the calibration/audit split. This notebook demonstrates coverage conditional on the pooled empirical template. Propagating uncertainty in that template is a separate hierarchical or nuisance-parameter problem discussed at the end.

Simulator toys remain fitted with `COMPRESSED_Q` from the hNDE likelihood. Replacing it by the simulator-template ratio would silently replace the approximate statistic by an oracle statistic and erase the calibration problem.

There is a subtler consistency catch. Exercise 5 saved ratios after dividing by the finite-reference means from its own five-million-event reference draw. Exercise 11 reconstructs its frozen likelihood with another reference draw, whose means differ slightly. Before histogramming simulator events, we therefore undo the saved Exercise 5 normalization and apply the current `RATIO_NORMALIZATION`. New artifacts may provide the exact values in `ratio_normalization.npz`; for the legacy committed run the notebook uses the six-decimal values printed in Exercise 5. The remaining rounding ambiguity is at the $10^{-6}$ scale, rather than the approximately $7\times10^{-4}$ uncorrected shift in $\log q$.


In [ ]:
simulator_templates = simulator_templates_from_exercise5(
    weights_path=HYBRID_DENSITY_DIR / "weights_asimov.npy",
    ratio_signal_path=HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy",
    ratio_background_path=HYBRID_DENSITY_DIR / "ratio_background_asimov.npy",
    log_q_edges=LOG_Q_EDGES,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    seed=SEED + 600,
    source_ratio_normalization=EXERCISE5_RATIO_NORMALIZATION,
    target_ratio_normalization=RATIO_NORMALIZATION,
)
SIM_SIGNAL_PROBABILITY = simulator_templates[
    "pooled_signal_probability"
]
SIM_BACKGROUND_PROBABILITY = simulator_templates[
    "pooled_background_probability"
]
split_signal_l1 = np.sum(np.abs(
    simulator_templates["calibration_signal_probability"]
    - simulator_templates["audit_signal_probability"]
))
split_background_l1 = np.sum(np.abs(
    simulator_templates["calibration_background_probability"]
    - simulator_templates["audit_background_probability"]
))
print(
    f"Held-out simulator events: "
    f"background={simulator_templates['n_background']:,}, "
    f"signal={simulator_templates['n_signal']:,}"
)
print(
    "Held-out weight/yield closure: "
    f"signal={simulator_templates['signal_yield_closure']:+.3%}, "
    f"background={simulator_templates['background_yield_closure']:+.3%}"
)
print(
    "Diagnostic half-template L1 differences (not used as two truths): "
    f"signal={split_signal_l1:.4f}, "
    f"background={split_background_l1:.4f}"
)
print(
    "Exercise 5 ratio normalizers from",
    EXERCISE5_NORMALIZATION_SOURCE,
    EXERCISE5_RATIO_NORMALIZATION,
)
print(
    "Saved-to-frozen log-q scale correction:",
    f"{simulator_templates['log_q_scale_correction']:+.6e}",
)
assert np.isclose(SIM_SIGNAL_PROBABILITY.sum(), 1.0)
assert np.isclose(SIM_BACKGROUND_PROBABILITY.sum(), 1.0)


## 6. Generate 100,000 simulator-calibration pseudo-experiments

The generating bin probabilities now come from the pooled held-out simulator template. The tested statistic remains the frozen hNDE profile-likelihood ratio. Consequently, any displacement in $\widehat\mu$, noncentrality in $t_\mu$, variance distortion, or tail mismatch is present in these toys and is available to the calibration map.

A pseudo-truth response map $g(\mu)$ could be learned separately to make the statistic more nearly central. We do not need it for coverage: a correct conditional CDF calibration absorbs the full distributional mismatch. Such a response correction may nevertheless improve smoothness and power, and is listed as an extension.


In [ ]:
simulator_calibration_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_pooled_calibration_{N_SIMULATOR_CALIBRATION_TOYS}"
    ),
    n_toys=N_SIMULATOR_CALIBRATION_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 700,
    mu_range=MU_RANGE,
    signal_probability=SIM_SIGNAL_PROBABILITY,
    background_probability=SIM_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
print(pd.DataFrame({
    "mu": simulator_calibration_toys["mu"],
    "mu_hat": simulator_calibration_toys["mu_hat"],
    "t_mu": simulator_calibration_toys["t_mu"],
}).describe(percentiles=[.01, .5, .95, .99]).to_string())

calibration_mu = simulator_calibration_toys["mu"].astype(np.float32)
calibration_y = np.log(
    simulator_calibration_toys["t_mu"].astype(np.float64) + T_OFFSET
)
below_y_grid = float(np.mean(calibration_y < Y_MIN))
above_y_grid = float(np.mean(calibration_y > Y_MAX))
print(
    "Simulator calibration toys outside Y_GRID (lower/upper):",
    f"{below_y_grid:.6%} / {above_y_grid:.6%}",
)
if below_y_grid > 0.0 or above_y_grid > 0.0:
    raise RuntimeError(
        "Simulator toys lie outside Y_GRID. Increase its support and "
        "rerun before training the PIT ratio; clipping would create "
        "an artificial atom at u=0 or u=1."
    )


## 7. The second ratio, now in the hNDE PIT coordinate

For every simulator pair $(\mu_i,t_i)$ define

$$
u_i^+=F_{\rm H}(t_i\mid\mu_i).
$$

At the **same** $\mu_i$, draw $u_i^-\sim\mathrm{Uniform}(0,1)$. A balanced classifier receives positive rows $(\mu_i,u_i^+)$ and negative rows $(\mu_i,u_i^-)$. Its population loss is

$$
\mathcal L_{\rm cal}
=-\mathbb E_{\pi(\mu)p_{\rm sim}^{U_0}(u\mid\mu)}\log D_{\rm cal}
 -\mathbb E_{\pi(\mu)\mathrm{Unif}(u)}\log(1-D_{\rm cal}),
$$

so balanced-class odds give

$$
\frac{D_{\rm cal}(u,\mu)}{1-D_{\rm cal}(u,\mu)}
=r_{\rm cal}(u,\mu)
=p_{\rm sim}^{U_0}(u\mid\mu).
$$

The matched $\mu$ values are essential: they make the joint proposal factor $\pi(\mu)$ cancel exactly and stop the classifier from learning an irrelevant parameter-marginal ratio. Each positive/negative pair also stays in the same train/validation group. Balanced class priors make the odds literal; a different class ratio would require the corresponding prior-odds factor, although a global constant would later cancel in conditional normalization. For float32 training only, values are clipped by $10^{-6}$ away from zero and one after first verifying that no genuine interior toy was mapped exactly to a boundary.

### Why this remains exactly the second density-ratio correction

For a continuous statistic, $u=F_{\rm H}(t\mid\mu)$ has Jacobian

$$
\frac{du}{dt}=p_{\rm H}(t\mid\mu).
$$

Therefore

$$
p_{\rm sim}^{U_0}(u\mid\mu)
=\left.
\frac{p_{\rm sim}^{T}(t\mid\mu)}
     {p_{\rm H}^{T}(t\mid\mu)}
\right|_{t=F_{\rm H}^{-1}(u\mid\mu)}.
$$

The proposed classifier is thus the same simulator-to-hNDE ratio as before, evaluated in a preconditioned coordinate. The reference is now bounded, exactly known, and trivial to sample. No $r_1/\widetilde Z_1$ importance weights or effective-sample-size problem remain, and exactly 5% of uniform reference samples lie above $u=0.95$.

A cache catch is easy to miss here: changing the pooled template, toy seed, clipping convention, or $F_{\rm H}$ grid changes every positive classifier row even if the checkpoint filename is unchanged. The PIT checkpoints therefore store a SHA-256 fingerprint of both class arrays, their paired group IDs, weights, seed, model configuration, and training configuration. Loading aborts on a mismatch and asks for a new `PIT_RUN_TAG`; silently accepting an old calibration network would invalidate the construction.


In [ ]:
calibration_u0_raw = conditional_cdf_values(
    hybrid1_grid["cdf"],
    MU_DENSITY_GRID,
    Y_GRID,
    calibration_mu,
    calibration_y,
)
exact_boundary_fraction = float(np.mean(
    (calibration_u0_raw <= 0.0) | (calibration_u0_raw >= 1.0)
))
print(f"Exact PIT-boundary fraction: {exact_boundary_fraction:.6%}")
if exact_boundary_fraction > 0.0:
    raise RuntimeError(
        "Interior simulator PIT values reached exactly 0 or 1. "
        "Inspect CDF support/interpolation before training."
    )
calibration_u0 = np.clip(
    calibration_u0_raw, PIT_EPS, 1.0 - PIT_EPS
).astype(np.float32)
pit_clipped_fraction = float(np.mean(
    (calibration_u0_raw < PIT_EPS)
    | (calibration_u0_raw > 1.0 - PIT_EPS)
))
print(
    f"PIT values moved by the numerical {PIT_EPS:g} clip: "
    f"{pit_clipped_fraction:.6%}"
)
if pit_clipped_fraction > 1.0e-3:
    raise RuntimeError(
        "More than 0.1% of simulator PIT values require numerical "
        "clipping. Inspect F_H support/tails or reduce PIT_EPS before "
        "training the calibration ratio."
    )

pit_rng = np.random.default_rng(SEED + 800)
calibration_u_reference = np.clip(
    pit_rng.uniform(0.0, 1.0, size=N_SIMULATOR_CALIBRATION_TOYS),
    PIT_EPS,
    1.0 - PIT_EPS,
).astype(np.float32)
calibration_positive = np.column_stack([
    calibration_mu, calibration_u0
])
calibration_negative = np.column_stack([
    calibration_mu, calibration_u_reference
])
if not np.array_equal(
    calibration_positive[:, 0], calibration_negative[:, 0]
):
    raise RuntimeError("The PIT classifier lost its matched mu design.")

paired_calibration_ids = np.arange(
    N_SIMULATOR_CALIBRATION_TOYS, dtype=np.int64
)
calibration_ratio_ensemble = []
for member in range(RATIO_ENSEMBLE_SIZE):
    print("\n" + "=" * 76)
    print(
        f"Training PIT calibration member "
        f"{member + 1}/{RATIO_ENSEMBLE_SIZE}"
    )
    print("=" * 76)
    calibration_ratio_ensemble.append(
        train_ratio_classifier(
            calibration_positive,
            calibration_negative,
            checkpoint=(
                CALIBRATION_RATIO_MODEL_DIR
                / f"r_cal_member{member}.pt"
            ),
            model_config=CORRECTION_MODEL_CONFIG,
            training_config=CORRECTION_TRAINING_CONFIG,
            device=device,
            seed=SEED + 810 + 100 * member,
            load_if_available=LOAD_IF_AVAILABLE,
            paired_group_ids=paired_calibration_ids,
            verify_checkpoint_data=True,
        )
    )
assert all(
    pack["history"].get("split_strategy") == "paired_groups"
    for pack in calibration_ratio_ensemble
)
assert not any(
    pack["history"].get("weighted_bce", False)
    for pack in calibration_ratio_ensemble
)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for member, pack in enumerate(calibration_ratio_ensemble):
    ax.plot(
        pack.get("history", {}).get("validation", []),
        label=f"member {member}",
    )
ax.axhline(np.log(2.0), color="black", ls="--", lw=1.2,
           label=r"$\log 2$")
ax.set(xlabel="Epoch", ylabel="Validation BCE",
       title="Simulator correction in PIT space")
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_training")
plt.show()


## 8. Normalize the PIT ratio and build the calibrated primitive

Classifier odds are not guaranteed to integrate to one for every $\mu$. We impose conditional normalization explicitly:

$$
Z_{\rm cal}(\mu)=\int_0^1r_{\rm cal}(v,\mu)\,dv,
\qquad
g_{\rm cal}(u\mid\mu)=\frac{r_{\rm cal}(u,\mu)}{Z_{\rm cal}(\mu)},
$$

$$
G(u\mid\mu)=\int_0^u g_{\rm cal}(v\mid\mu)\,dv.
$$

This is a one-dimensional quadrature regardless of the dimension of the conditioning parameter. The final raw-statistic density and CDF are

$$
p_{\rm cal}(t\mid\mu)
=p_{\rm H}(t\mid\mu)
 g_{\rm cal}\!\left(F_{\rm H}(t\mid\mu)\mid\mu\right),
$$

$$
F_{\rm cal}(t\mid\mu)
=G\!\left(F_{\rm H}(t\mid\mu)\mid\mu\right).
$$

The global scale of classifier odds cancels in $Z_{\rm cal}$. A $\mu$-dependent scale error also cancels row by row. Shape errors in $u$ do not cancel and are precisely what the independent auditor tests.

The following PIT histograms use the calibration sample itself and are therefore training/resubstitution diagnostics, not evidence of coverage. Only the later untouched ensemble is an audit.


In [ ]:
U_GRID = np.linspace(0.0, 1.0, PIT_QUADRATURE_POINTS)
calibration_grid = conditional_ratio_grid(
    calibration_ratio_ensemble,
    MU_DENSITY_GRID,
    U_GRID,
    max_abs_log_ratio=LOG_RATIO_CLIP,
)
print(
    "raw log Z_cal(mu) quantiles:",
    np.quantile(
        calibration_grid["log_normalization"],
        [0, .01, .5, .99, 1],
    ),
)
print(
    "PIT-ratio logit range / clipped fraction:",
    calibration_grid["ratio_log_range"],
    f"{calibration_grid['ratio_clip_fraction']:.4%}",
)
if (
    np.max(np.abs(calibration_grid["cdf"][:, 0])) > 1.0e-10
    or np.max(np.abs(calibration_grid["cdf"][:, -1] - 1.0))
    > 1.0e-10
    or np.min(np.diff(calibration_grid["cdf"], axis=1)) < -1.0e-10
):
    raise RuntimeError("G is not a valid conditional CDF.")

def evaluate_calibrated_pit(mu, t_mu):
    """Return (U0, Ucal) for arbitrary interior toy/statistic rows."""
    mu = np.asarray(mu, dtype=np.float64).reshape(-1)
    t_mu = np.asarray(t_mu, dtype=np.float64).reshape(-1)
    if len(mu) != len(t_mu) or np.any(t_mu < 0.0):
        raise ValueError("mu/t_mu must be matched and t_mu non-negative.")
    if np.any(mu <= MU_RANGE[0]) or np.any(mu >= MU_RANGE[1]):
        raise ValueError("Use the empirical rules at exact endpoints.")
    y = np.log(t_mu + T_OFFSET)
    if np.any(y < Y_MIN) or np.any(y > Y_MAX):
        raise ValueError("Statistic values lie outside the frozen Y_GRID.")
    u0 = conditional_cdf_values(
        hybrid1_grid["cdf"], MU_DENSITY_GRID, Y_GRID, mu, y
    )
    u_cal = conditional_cdf_values(
        calibration_grid["cdf"], MU_DENSITY_GRID, U_GRID, mu, u0
    )
    return u0, u_cal


def evaluate_calibrated_statistic(mu, t_mu):
    """Map Ucal to a chi-square-looking scalar statistic."""
    _, u_cal = evaluate_calibrated_pit(mu, t_mu)
    return chi2.ppf(np.clip(u_cal, 1.0e-12, 1.0 - 1.0e-12), df=1)


_, calibration_u_cal = evaluate_calibrated_pit(
    calibration_mu, simulator_calibration_toys["t_mu"]
)

selected_mu = [0.25, 1.0, 2.0, 2.75]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for mu_value in selected_mu:
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    axes[0].plot(
        U_GRID,
        calibration_grid["density"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
    axes[1].plot(
        U_GRID,
        calibration_grid["cdf"][index],
        lw=1.8,
        label=rf"$\mu={mu_value:g}$",
    )
axes[0].axhline(1.0, color="black", ls="--", lw=1.2,
                label="Uniform density")
axes[1].plot(U_GRID, U_GRID, "k--", lw=1.2,
             label="Identity / no correction")
axes[0].set(xlabel=r"hNDE PIT $u_0$", ylabel=r"$g_{\rm cal}(u_0\mid\mu)$",
            title="Learned simulator/hNDE ratio")
axes[1].set(xlabel=r"hNDE PIT $u_0$", ylabel=r"$G(u_0\mid\mu)$",
            title="Conditional calibration map")
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.tight_layout()
export_exercise11_figure(fig, "pit_ratio_and_primitive")
plt.show()

bins = np.linspace(0.0, 1.0, 41)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
axes[0].hist(calibration_u0_raw, bins=bins, density=True,
             histtype="step", lw=2, color="C1")
axes[1].hist(calibration_u_cal, bins=bins, density=True,
             histtype="step", lw=2, color="C0")
for ax in axes:
    ax.axhline(1.0, color="black", ls="--", lw=1.2)
    ax.set(xlim=(0, 1), xlabel="PIT value", ylabel="Density")
    ax.grid(alpha=.2)
axes[0].set_title(r"Before calibration: $U_0=F_{\rm H}(T\mid\mu)$")
axes[1].set_title(r"After calibration: $U_{\rm cal}=G(U_0\mid\mu)$")
fig.tight_layout()
export_exercise11_figure(fig, "pit_calibration_training_closure")
plt.show()


## 9. Conditional quantiles, endpoint atoms, and Neyman inversion

The calibrated $\gamma$ quantile can be obtained without sampling the learned density:

$$
u_\gamma(\mu)=G^{-1}(\gamma\mid\mu),
\qquad
c_\gamma(\mu)
=F_{\rm H}^{-1}\!\left(u_\gamma(\mu)\mid\mu\right).
$$

We compare these curves with the uncalibrated hNDE quantiles $F_{\rm H}^{-1}(\gamma\mid\mu)$ and with direct, coarse binned simulator quantiles. The latter are a useful sanity check but use the same calibration ensemble, so they are not an audit.

### The $\mu=0$ catch

All compressed Poisson toy statistics are formally discrete, so the textbook non-randomized PIT need not be exactly uniform at finite event counts. In the interior the support is sufficiently fine that this discreteness is negligible at the resolution of the exercise; at $\mu=0$ it is not. The continuous proposal also draws no point exactly at $\mu=0$ or $3$, and the constraint $\widehat\mu\geq0$ creates a genuine point mass at $t_0=0$. A continuous flow followed by an absolutely continuous density ratio cannot represent that singular atom. The formal exact remedy for any discrete law is the randomized PIT

$$
U=F(T^-\mid\mu)+V\,[F(T\mid\mu)-F(T^-\mid\mu)],
\qquad V\sim\mathrm{Uniform}(0,1),
$$

but that produces a randomized test. For this tutorial we instead generate explicit endpoint ensembles and use the finite-sample conservative order statistic

$$
k=\left\lceil\gamma(n+1)\right\rceil,
\qquad c_\gamma=T_{(k)}.
$$

Ties can make this non-randomized endpoint test conservative. The empirical endpoint quantiles are treated piecewise and are **not** inserted into the smooth interior cutoff interpolant, so the $\mu=0$ atom does not distort nearby interior cutoffs. The continuous $F_{\rm H}$ and $G$ tables still contain endpoint rows, which help interpolate arbitrarily close to a boundary. The helper defined below implements exactly this piecewise observed-data rule: calibrated PIT inversion in the open interval and empirical cutoffs at the two exact endpoints.


In [ ]:
calibration_u0_quantiles = conditional_quantiles(
    calibration_grid["cdf"], U_GRID, QUANTILE_LEVELS
)
calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    calibration_u0_quantiles,
)
calibrated_quantile_t = np.maximum(
    np.exp(calibrated_quantile_y) - T_OFFSET, 0.0
)

coarse_u_grid = U_GRID[::2]
coarse_calibration_density = calibration_grid["density"][:, ::2]
coarse_calibration_normalization = trapezoid(
    coarse_calibration_density, x=coarse_u_grid, axis=1
)
coarse_calibration_cdf = cumulative_trapezoid(
    coarse_calibration_density,
    x=coarse_u_grid,
    axis=1,
    initial=0.0,
) / coarse_calibration_normalization[:, None]
coarse_calibration_u0_quantiles = conditional_quantiles(
    coarse_calibration_cdf, coarse_u_grid, QUANTILE_LEVELS
)
coarse_calibrated_quantile_y = conditional_row_quantiles(
    hybrid1_grid["cdf"],
    Y_GRID,
    coarse_calibration_u0_quantiles,
)
u_resolution_shift = float(np.max(np.abs(
    coarse_calibrated_quantile_y - calibrated_quantile_y
)))
u_resolution_tolerance = max(0.02, 4.0 * np.max(np.diff(Y_GRID)))
print(
    "Full/half u-grid maximum final-quantile shift / tolerance:",
    u_resolution_shift, u_resolution_tolerance,
)
if u_resolution_shift > u_resolution_tolerance:
    raise RuntimeError(
        "The calibrated conditional quantiles are not stable when "
        "the u-grid resolution is halved. Increase "
        "PIT_QUADRATURE_POINTS."
    )

calibrated_cdf_grid = np.asarray([
    np.interp(f_h_row, U_GRID, g_row)
    for f_h_row, g_row in zip(
        hybrid1_grid["cdf"], calibration_grid["cdf"]
    )
])
composed_quantile_y = conditional_quantiles(
    calibrated_cdf_grid, Y_GRID, QUANTILE_LEVELS
)
composition_difference = np.max(np.abs(
    composed_quantile_y - calibrated_quantile_y
))
print(
    "Maximum nested/composed quantile difference in y:",
    composition_difference,
)
if composition_difference > 2.0 * np.max(np.diff(Y_GRID)):
    raise RuntimeError("Nested and composed conditional CDFs disagree.")

index_95 = int(np.flatnonzero(
    np.isclose(QUANTILE_LEVELS, .95)
)[0])
critical_hnde_grid = hybrid1_quantile_t[:, index_95]
critical_calibrated_grid = calibrated_quantile_t[:, index_95]
critical_calibrated_smooth = PchipInterpolator(
    MU_DENSITY_GRID, critical_calibrated_grid, extrapolate=False
)

anchor_hnde = {}
anchor_simulator = {}
anchor_hnde_quantiles = []
anchor_simulator_quantiles = []
for anchor_mu in ANCHOR_MUS:
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_hnde[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=CACHE_DIR / f"anchor_hnde_{key}_{N_ANCHOR_TOYS}",
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 900 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=HNDE_SIGNAL_PROBABILITY,
        background_probability=HNDE_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )
    anchor_simulator[anchor_mu] = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_pooled_calibration_{key}_{N_ANCHOR_TOYS}"
        ),
        n_toys=N_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 910 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_SIGNAL_PROBABILITY,
        background_probability=SIM_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )
    anchor_hnde_quantiles.append(conservative_empirical_quantile(
        anchor_hnde[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
    anchor_simulator_quantiles.append(conservative_empirical_quantile(
        anchor_simulator[anchor_mu]["t_mu"], QUANTILE_LEVELS
    ))
anchor_hnde_quantiles = np.asarray(anchor_hnde_quantiles)
anchor_simulator_quantiles = np.asarray(anchor_simulator_quantiles)

def neyman_accepts_95(mu, t_mu):
    """Apply the endpoint-aware 95% observed-data decision rule."""
    mu, t_mu = np.broadcast_arrays(
        np.asarray(mu, dtype=np.float64),
        np.asarray(t_mu, dtype=np.float64),
    )
    output_shape = mu.shape
    mu = mu.reshape(-1)
    t_mu = t_mu.reshape(-1)
    if np.any(t_mu < 0.0) or not np.isfinite(t_mu).all():
        raise ValueError("t_mu must be finite and non-negative.")
    if (
        not np.isfinite(mu).all()
        or np.any(mu < MU_RANGE[0])
        or np.any(mu > MU_RANGE[1])
    ):
        raise ValueError("mu lies outside the calibrated parameter range.")
    accepted = np.empty(len(mu), dtype=bool)
    assigned = np.zeros(len(mu), dtype=bool)
    interior = (mu > MU_RANGE[0]) & (mu < MU_RANGE[1])
    if np.any(interior):
        _, u_cal = evaluate_calibrated_pit(mu[interior], t_mu[interior])
        accepted[interior] = u_cal <= .95
        assigned[interior] = True
    for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
        endpoint = mu == anchor_mu
        accepted[endpoint] = (
            t_mu[endpoint]
            <= anchor_simulator_quantiles[anchor_index, index_95]
        )
        assigned[endpoint] = True
    if not np.all(assigned):
        raise ValueError("mu lies outside the calibrated parameter range.")
    return accepted.reshape(output_shape)

calibration_quantile_edges = np.linspace(*MU_RANGE, 21)
calibration_quantile_centers = 0.5 * (
    calibration_quantile_edges[:-1] + calibration_quantile_edges[1:]
)
calibration_bin_index = np.clip(
    np.digitize(calibration_mu, calibration_quantile_edges) - 1,
    0,
    len(calibration_quantile_centers) - 1,
)
direct_calibration_q95 = np.asarray([
    conservative_empirical_quantile(
        simulator_calibration_toys["t_mu"][calibration_bin_index == i],
        .95,
    )
    for i in range(len(calibration_quantile_centers))
])

np.savez_compressed(
    PIT_CACHE_DIR / "conditional_pit_calibration.npz",
    mu=MU_DENSITY_GRID,
    levels=QUANTILE_LEVELS,
    hnde=hybrid1_quantile_t,
    simulator_calibrated=calibrated_quantile_t,
    endpoint_mu=ANCHOR_MUS,
    endpoint_hnde=anchor_hnde_quantiles,
    endpoint_simulator=anchor_simulator_quantiles,
    u_grid=U_GRID,
    calibration_cdf=calibration_grid["cdf"],
)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
colors = plt.cm.viridis(np.linspace(.12, .9, len(QUANTILE_LEVELS)))
interior_grid = slice(1, -1)
for level, color, before, after in zip(
    QUANTILE_LEVELS,
    colors,
    hybrid1_quantile_t.T,
    calibrated_quantile_t.T,
):
    linewidth = 2.8 if np.isclose(level, .95) else 1.25
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], before[interior_grid],
        color=color, ls="--", lw=linewidth,
        label=(rf"{level:.0%} hNDE" if np.isclose(level, .95)
               else rf"{level:.0%}"),
    )
    axes[0].plot(
        MU_DENSITY_GRID[interior_grid], after[interior_grid],
        color=color, lw=linewidth,
        label=(rf"{level:.0%} calibrated"
               if np.isclose(level, .95) else None),
    )
    axes[1].plot(
        MU_DENSITY_GRID[interior_grid],
        (after - before)[interior_grid],
        color=color, lw=linewidth, label=rf"{level:.0%}",
    )
axes[0].scatter(
    ANCHOR_MUS,
    anchor_simulator_quantiles[:, index_95],
    marker="s", s=42, color="C3", zorder=5,
    label="empirical endpoint 95%",
)
axes[0].scatter(
    calibration_quantile_centers,
    direct_calibration_q95,
    marker="o", s=18, facecolors="none", edgecolors="0.25",
    label="binned simulator 95%",
)
axes[0].set(
    xlabel=r"$\mu$", ylabel=r"conditional quantile of $t_\mu$",
    title="Before and after PIT-ratio calibration",
)
axes[1].axhline(0.0, color="black", ls=":", lw=1)
axes[1].set(
    xlabel=r"$\mu$", ylabel="calibrated − hNDE quantile",
    title="Calibration displacement",
)
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_quantile_corrections")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0),
                         sharex=True, sharey=True)
t_grid = np.maximum(np.exp(Y_GRID) - T_OFFSET, 0.0)
for ax, mu_value in zip(axes.flat, [0.25, 1.0, 2.0, 2.75]):
    index = int(np.argmin(np.abs(MU_DENSITY_GRID - mu_value)))
    ax.plot(t_grid, hybrid1_grid["cdf"][index], ls="--", lw=2,
            label="hNDE hybrid")
    ax.plot(t_grid, calibrated_cdf_grid[index], lw=2,
            label="PIT-ratio calibrated")
    ax.axhline(.95, color="0.4", ls=":", lw=1)
    ax.set_xlim(
        0, max(8.0, float(critical_calibrated_grid[index]) * 1.5)
    )
    ax.set_title(rf"$\mu={mu_value:g}$")
    ax.grid(alpha=.2)
for ax in axes[-1]:
    ax.set_xlabel(r"$t_\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel("Conditional CDF")
axes[0, 0].legend()
fig.tight_layout()
export_exercise11_figure(fig, "conditional_pit_cdf_primitives")
plt.show()


## 10. LF2I-style independent coverage auditor

We now freeze $F_{\rm H}$, the PIT-ratio ensemble, $G$, and all numerical grids. A completely unused simulator ensemble is generated from the same pooled simulator law with a different random seed. Each audit toy is mapped through

$$
U_{0,i}=F_{\rm H}(t_i\mid\mu_i),
\qquad
U_{{\rm cal},i}=G(U_{0,i}\mid\mu_i),
$$

and labeled

$$
W_i=\mathbb I[U_{{\rm cal},i}\leq0.95].
$$

A probabilistic network receives **only** $\mu_i$ and minimizes ordinary, unweighted BCE at the natural 95/5 class frequency. Its population target is the local coverage function

$$
a^*(\mu)=\Pr_{\rm sim}(W=1\mid\mu).
$$

We also show direct equal-width-bin estimates with Wilson intervals, preventing a too-smooth auditor from hiding localized failures. This is an empirical audit, not a proof: it resolves deviations only at the scale supported by 100,000 toys and the auditor architecture. Most importantly, if the audit result is used to modify the construction, this ensemble is no longer an audit; a new untouched ensemble must be generated.

The direct pivot decision is canonical. We also compare it with the interpolated raw-$t$ cutoff as a numerical diagnostic. Small disagreements can arise because the cutoff curve is tabulated and interpolated twice; they do not enter the reported pivot coverage.


In [ ]:
simulator_audit_toys = run_cached_toy_ensemble(
    cache_dir=(
        PIT_CACHE_DIR
        / f"simulator_pooled_audit_{N_SIMULATOR_AUDIT_TOYS}"
    ),
    n_toys=N_SIMULATOR_AUDIT_TOYS,
    batch_size=TOY_BATCH_SIZE,
    seed=SEED + 1000,
    mu_range=MU_RANGE,
    signal_probability=SIM_SIGNAL_PROBABILITY,
    background_probability=SIM_BACKGROUND_PROBABILITY,
    lam_signal=LAM_SIG,
    lam_background=LAM_BKG,
    likelihood_q=COMPRESSED_Q,
    fit_batch=fit_toy_batch_numpy,
    fit_fingerprint=TOY_FIT_FINGERPRINT,
)
audit_mu = simulator_audit_toys["mu"].astype(np.float64)
audit_t = simulator_audit_toys["t_mu"].astype(np.float64)
audit_u0, audit_u_cal = evaluate_calibrated_pit(audit_mu, audit_t)
covered_hnde = audit_u0 <= .95
covered_calibrated = audit_u_cal <= .95
if not np.array_equal(
    neyman_accepts_95(audit_mu, audit_t), covered_calibrated
):
    raise RuntimeError(
        "The endpoint-aware helper disagrees with the interior PIT rule."
    )

audit_critical_calibrated = critical_calibrated_smooth(audit_mu)
covered_by_cutoff = audit_t <= audit_critical_calibrated
cutoff_disagreement = float(np.mean(
    covered_by_cutoff != covered_calibrated
))
print(f"Pivot/cutoff decision disagreement: {cutoff_disagreement:.4%}")
if cutoff_disagreement > 2.0e-3:
    print(
        "WARNING: the auxiliary interpolated cutoff disagrees with "
        "more than 0.2% of direct pivot decisions. The pivot audit "
        "remains valid, but refine the mu/y/u grids before using the "
        "smooth cutoff curve as a numerical substitute."
    )

audit_t_cal = evaluate_calibrated_statistic(audit_mu, audit_t)
print(f"Global hNDE-only coverage:          {covered_hnde.mean():.4%}")
print(f"Global PIT-calibrated coverage:     {covered_calibrated.mean():.4%}")
print(
    "Equivalent chi-square decision coverage:",
    f"{np.mean(audit_t_cal <= chi2.ppf(.95, df=1)):.4%}",
)

coverage_auditor = train_coverage_auditor(
    audit_mu,
    covered_calibrated,
    checkpoint=PIT_MODEL_DIR / "coverage_auditor_pit_95cl.pt",
    model_config=AUDITOR_MODEL_CONFIG,
    training_config=AUDITOR_TRAINING_CONFIG,
    device=device,
    seed=SEED + 1010,
    load_if_available=LOAD_IF_AVAILABLE,
)
auditor_grid = coverage_auditor_probability(
    coverage_auditor, MU_DENSITY_GRID
)
null_bce = -0.95 * np.log(.95) - 0.05 * np.log(.05)
print(f"Bernoulli(0.95) null BCE: {null_bce:.6f}")
print(
    "Auditor predicted-coverage range:",
    f"[{auditor_grid.min():.4%}, {auditor_grid.max():.4%}]",
)


In [ ]:
coverage_edges = np.linspace(*MU_RANGE, 21)
binned_before = binned_coverage(
    audit_mu, covered_hnde, edges=coverage_edges
)
binned_after = binned_coverage(
    audit_mu, covered_calibrated, edges=coverage_edges
)

anchor_audit_rows = []
for anchor_index, anchor_mu in enumerate(ANCHOR_MUS):
    key = f"mu_{anchor_mu:g}".replace(".", "p")
    anchor_audit = run_cached_toy_ensemble(
        cache_dir=(
            PIT_CACHE_DIR
            / f"anchor_simulator_pooled_audit_{key}_{N_AUDIT_ANCHOR_TOYS}"
        ),
        n_toys=N_AUDIT_ANCHOR_TOYS,
        batch_size=TOY_BATCH_SIZE,
        seed=SEED + 1100 + int(100 * anchor_mu),
        mu_range=MU_RANGE,
        fixed_mu=float(anchor_mu),
        signal_probability=SIM_SIGNAL_PROBABILITY,
        background_probability=SIM_BACKGROUND_PROBABILITY,
        lam_signal=LAM_SIG,
        lam_background=LAM_BKG,
        likelihood_q=COMPRESSED_Q,
        fit_batch=fit_toy_batch_numpy,
        fit_fingerprint=TOY_FIT_FINGERPRINT,
    )
    endpoint_critical = float(
        anchor_simulator_quantiles[anchor_index, index_95]
    )
    endpoint_covered = neyman_accepts_95(
        np.full(N_AUDIT_ANCHOR_TOYS, anchor_mu),
        anchor_audit["t_mu"],
    )
    successes = int(endpoint_covered.sum())
    lower, upper = wilson_interval(successes, len(endpoint_covered))
    anchor_audit_rows.append({
        "mu": anchor_mu,
        "critical_value": endpoint_critical,
        "coverage": float(endpoint_covered.mean()),
        "wilson_lower": float(lower),
        "wilson_upper": float(upper),
        "zero_mass": float(np.mean(
            anchor_audit["t_mu"] <= 1.0e-12
        )),
    })
anchor_audit_table = pd.DataFrame(anchor_audit_rows)
display(anchor_audit_table.style.format(precision=5).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.axhline(.95, color="black", ls="--", lw=1.5,
           label="Nominal 95%")
ax.plot(MU_DENSITY_GRID, auditor_grid, color="C3", lw=2.3,
        label="BCE coverage auditor")
ax.errorbar(
    binned_before["center"], binned_before["coverage"],
    yerr=[
        binned_before["coverage"] - binned_before["lower"],
        binned_before["upper"] - binned_before["coverage"],
    ],
    fmt="o", ms=4, color="0.5", alpha=.75,
    label="hNDE PIT without simulator calibration",
)
ax.errorbar(
    binned_after["center"], binned_after["coverage"],
    yerr=[
        binned_after["coverage"] - binned_after["lower"],
        binned_after["upper"] - binned_after["coverage"],
    ],
    fmt="o", ms=5, color="C0", label="PIT-ratio calibrated",
)
ax.errorbar(
    anchor_audit_table["mu"], anchor_audit_table["coverage"],
    yerr=[
        anchor_audit_table["coverage"]
        - anchor_audit_table["wilson_lower"],
        anchor_audit_table["wilson_upper"]
        - anchor_audit_table["coverage"],
    ],
    fmt="s", ms=6, color="C2",
    label="independent endpoint anchors",
)
ax.set(
    xlim=MU_RANGE,
    xlabel=r"true $\mu$",
    ylabel="Conditional coverage",
    title="Independent LF2I coverage audit",
)
ax.set_ylim(min(.90, binned_before["lower"].min() - .005), 1.005)
ax.grid(alpha=.2)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
export_exercise11_figure(fig, "lf2i_pit_coverage_audit")
plt.show()

coverage_summary = pd.DataFrame({
    "method": ["hNDE PIT", "simulator-calibrated PIT"],
    "global_coverage": [
        covered_hnde.mean(), covered_calibrated.mean()
    ],
    "minimum_binned_coverage": [
        binned_before["coverage"].min(),
        binned_after["coverage"].min(),
    ],
    "maximum_binned_coverage": [
        binned_before["coverage"].max(),
        binned_after["coverage"].max(),
    ],
})
display(coverage_summary.style.format(precision=5).hide(axis="index"))


## Interpretation: what has and has not been calibrated

This exercise separates roles that are easy to conflate:

- The frozen hNDE likelihood defines the **ordering statistic**. Improving the event-level density model should generally improve power and shorten confidence intervals.
- The first spline-plus-ratio construction estimates the hNDE sampling CDF $F_{\rm H}$. It supplies a powerful baseline using inexpensive toys.
- The PIT-ratio map estimates the remaining simulator discrepancy. It calibrates bias in $\widehat\mu$, noncentrality, variance distortion, skewness, and tail shape together because it targets the entire conditional CDF.
- Calibration does **not** make $\widehat\mu$ unbiased and does not repair a weak ordering statistic. An imperfect statistic can have correct size after calibration while losing power or producing longer/asymmetric intervals.
- The LF2I auditor is independent of construction. A flat auditor near 95%, supported by binned Wilson intervals and endpoint anchors, means no conditional coverage departure is statistically resolved at this audit's resolution. It is not a mathematical proof.

The confidence set for observed data $\mathcal D_{\rm obs}$ is obtained by inversion:

$$
\mathcal C_{0.95}(\mathcal D_{\rm obs})
=\left\{\mu\in(0,3):
G\!\left(F_{\rm H}(t_\mu(\mathcal D_{\rm obs})\mid\mu)\mid\mu\right)
\leq0.95\right\},
$$

with the explicit empirical endpoint rules used at $\mu=0$ and $3$. The equivalent cutoff form is

$$
t_\mu(\mathcal D_{\rm obs})
\leq F_{\rm H}^{-1}\!\left(G^{-1}(0.95\mid\mu)\mid\mu\right).
$$


## The finite-template limitation

The audit in this notebook establishes empirical coverage **conditional on the pooled held-out simulator template**. This is the appropriate first test of the calibration algorithm: calibration and audit toys are independent draws from one fixed data-generating law.

A production analysis must also represent uncertainty in that finite template. There are several distinct coverage targets:

1. **Conditional on one template:** the target demonstrated here.
2. **Average over template uncertainty:** draw a bootstrap or posterior template realization inside both calibration and audit pseudo-experiments. This defines a hierarchical simulator law.
3. **Uniformly conservative over template variations:** calibrate to the worst-case cutoff over an allowed nuisance set. This generally costs power.
4. **Explicit nuisance augmentation:** include template/MC-statistical nuisance parameters in the parameter vector supplied to the calibration ratio and in the Neyman construction.

Calibrating on one noisy half-template and auditing on a different half-template mixes the first and second questions and cannot guarantee 95% coverage. Their observed difference is useful evidence that template uncertainty matters, but it is not a failure of conditional CDF calibration.


## Why the construction extends to high-dimensional parameters

Replace the scalar $\mu$ by $\boldsymbol\theta\in\mathbb R^d$. The models become

$$
q_\phi(y\mid\boldsymbol\theta),
\qquad
r_1(y,\boldsymbol\theta),
\qquad
r_{\rm cal}(u,\boldsymbol\theta).
$$

The simulator classifier input has dimension $d+1$, but its conditional normalizer remains

$$
Z_{\rm cal}(\boldsymbol\theta)
=\int_0^1r_{\rm cal}(u,\boldsymbol\theta)\,du,
$$

a one-dimensional integral. In high dimensions one would evaluate this quadrature on demand for batches of requested $\boldsymbol\theta$ values rather than construct a Cartesian parameter grid. The same is true for the $t$ primitive: the ordering statistic is scalar even when the tested parameter is not.

This solves the density-scale and numerical-integration problem; it does not abolish the simulator curse of dimensionality. The proposal $\pi(\boldsymbol\theta)$ must cover the deployment region, the networks must interpolate adequately, and the coverage auditor remains essential. Nuisance parameters may be included in $\boldsymbol\theta$ when the desired guarantee is conditional on them, or marginalized according to a precisely stated ensemble when the desired guarantee is averaged over them.


## Suggested exercises

1. Increase the simulator-calibration sample and study convergence of $G(u\mid\mu)$, $c_{0.95}(\mu)$, and the auditor.
2. Compare the full PIT-density correction with direct 95% conditional quantile regression, the canonical LF2I branch. The ratio learns all confidence levels at once; a tail-specific regressor may be more efficient for one fixed level.
3. Learn a monotone pseudo-truth response map $g(\mu)$, define a response-corrected statistic, and test whether the PIT correction becomes smaller and the resulting intervals gain power.
4. Add proposal mass near boundaries, or train a spike-and-slab statistic model, and compare with the empirical endpoint construction.
5. Bootstrap the held-out simulator template inside the ensemble and distinguish conditional, average, and worst-case coverage targets.
6. Double both the $y$ and $u$ quadrature resolutions and verify stability of the 95% cutoff and pivot/cutoff decisions.
7. Invert the calibrated pivot for several observed pseudo-datasets and compare confidence-set length before and after simulator calibration.
